# Reinforcement Learning - Lab 2
### J. Martinet

## 1) Optional: finish the implementation of TicTacToe

Remember that the general definition of the TD update rule is:

$$ V(s_t) \leftarrow V(s_t) + \alpha[ V(s_{t+1}) - V(s_{t}] $$

(We will come back to this later.)

## 2) Excercices

### A) Examples
Devise three example tasks of your own that fit into the reinforcement learning framework, identifying for each its states, actions, and rewards. Make the three examples as different from each other as possible.
The framework is abstract and exible and can be applied in many different ways. Stretch its limits in some way in at least one of your examples.

#### Your answer here here:
...

## Example 1: Robot Cleaning a Room (Classic Control Task)

Task description:
A small robot moves inside a room and tries to clean all dirty tiles while avoiding obstacles.

### States

The robot’s current position on a grid (e.g., (x, y))

Whether the current tile is clean or dirty

Example state:
((2,3), dirty)

### Actions

Move up

Move down

Move left

Move right

Clean the current tile

 ### Rewards

+10 for cleaning a dirty tile

-1 for each movement (to encourage efficiency)

-5 if the robot hits a wall or obstacle

0 if it cleans an already clean tile

### Goal

Learn a policy that cleans the entire room using the fewest steps.

## Example 2: Student Studying for an Exam (Abstract / Non-Physical Task)

## Task description:
Agent will create a study plain.
An agent represents a student deciding how to study over several days before an exam.

States

Knowledge level: {low, medium, high}

Energy level: {tired, normal}

Example state:
(medium knowledge, tired)

### Actions

Study

Take a break

Sleep

### Rewards

+5 for studying when energy is normal

-3 for studying while tired

+2 for sleeping when tired

+20 if the exam is passed at the end

-10 if the exam is failed

### Goal

Learn when to study, rest, or sleep to maximize exam success.

## Example 3: Music Recommendation System (Stretching the Framework)

### Task description:
An agent recommends songs to a user and learns from the user’s reactions.

### States

User’s recent listening history (e.g., last 3 genres)

Time of day (morning, afternoon, night)

Example state:
([pop, pop, jazz], night)

### Actions

Recommend pop

Recommend jazz

Recommend rock

Recommend classical

### Rewards

+1 if the user listens to the full song

+3 if the user likes the song

-2 if the user skips the song immediately

### Goal

Maximize long-term user satisfaction and engagement.

### B) Agent-environment boundary
Consider the problem of driving. You could define the actions
in terms of the accelerator, steering wheel, and brake, that is, where your body meets the machine.

Or you could define them farther out -- say, where the rubber meets the road, considering your actions to be tire torques.

Or you could define them farther in -- say, where your brain meets your body, the actions being muscle twitches to control your limbs.

Or you could go to a really high level and say that your actions are your choices of where to drive.

What is the right level, the right place to draw the line between agent and environment? On what basis is one location of the line to be preferred over another? Is there any fundamental reason for preferring one location over another, or is it a free choice?

#### Your answer here here:
...

### The Short Answer
The boundary is drawn based on **control**.
* Anything the agent can change arbitrarily *right now* is part of the **Agent**.
* Anything that follows physical laws, requires a reaction time, or is outside direct control is the **Environment**.

### Detailed Explanation
There is no single "correct" place to draw the line; it depends on what you want to learn. However, the general rule of thumb is: **The Agent is the decision-making unit.**

**Scenario 1: High Level (Route Choice)**
* **Goal:** To learn the fastest route to work.
* **Agent:** You (the mind).
* **Environment:** The car, the roads, the traffic, and even your own hands turning the wheel.
* **Why?** The decision is "Turn Left," but the mechanics of turning are assumed to work automatically.

**Scenario 2: Low Level (Muscle Twitches)**
* **Goal:** To learn how to walk using a prosthetic leg.
* **Agent:** The electrical signals in the nerve ending.
* **Environment:** The motors in the leg, the wind, gravity.
* **Why?** You need to learn exactly how much voltage to send to move the motor.

### Beginner Analogy
Think of playing a video game.
* **You (The Agent):** You press the buttons ($X$, $O$, $\Delta$, $\square$).
* **The Environment:** The console, the TV screen, and the game code.

Even though you "control" the character, if the character slips on ice, that is the environment reacting to your action. You cannot "choose" to not slip; you can only "choose" to move the joystick.

---

### C) Lazy robot
Imagine that you are designing a robot to run a maze. You decide
to give it a reward of +1 for escaping from the maze and a reward of zero at all other times.

The task seems to break down naturally into episodes (the
successive runs through the maze) so you decide to treat it as an episodic task, where the goal is to maximize expected total reward.

After running the learning agent for a while, you find that it is showing no improvement in escaping from the maze. What is going wrong? Have you effectively communicated to the agent what you want it to achieve?

#### Your answer here here:
...
### What is going wrong?
You have created a problem of **sparse rewards** and lack of **urgency**.

If the robot gets a reward of $+1$ for escaping, and $0$ for wandering around:
1.  Escaping in 10 steps gives a total reward of $+1$.
2.  Escaping in 1,000,000 steps gives a total reward of $+1$.

To the robot, these two outcomes are equal. Since finding the exit is hard, and wandering is easy, it may just wander forever because it feels no pressure to finish quickly. It hasn't learned that "time is money."

### How to fix it
You need to communicate that "escaping *quickly* is better than just escaping."

1.  **Discounting:** Use a discount factor ($\gamma < 1$). This makes a $+1$ reward *now* worth more than a $+1$ reward *later*.
2.  **Living Penalty (Step Cost):** Change the reward for "all other times" from $0$ to $-0.1$. Now, every step the robot takes is painful. It will race to the exit to stop the accumulation of negative rewards.

---



### D Gridworld (from Sutton and Barto)

The figure below shows a rectangular gridworld representation of a simple finite MDP.

The cells of the grid correspond to the states of the environment.

At each cell, four actions are possible: north, south, east, and west, which deterministically cause the agent to move one cell in the respective direction on the grid.

Actions that would take the agent o↵ the grid leave its location unchanged, but also result in a reward of -1.

Other actions result in a reward of 0, except those that move the agent out of the special states A and B. From state A, all four actions yield a reward of +10 and take the agent to A'. From state B, all actions yield a reward of +5 and take the agent to B'.

![Grid world](gridworld.png)

Suppose the agent selects all four actions with equal probability in all states. 

The right part of the figure shows the value function, $v_\pi$, for this policy, for the discounted reward case with $\gamma$ = 0.9. This value function was computed by solving the system of linear equations (Bellman equation).

Notice the negative values near the lower edge; these are the result of the high probability of hitting the edge of the grid there under the random policy. State A is the best state to be in under this policy, but its expected return is less than 10, its immediate reward, because from A the agent is taken to A', from which it is likely to run into the edge of the grid. State B, on the other hand, is valued more than 5, its immediate reward, because from B the agent is taken to B', which has a positive value. From B' the expected penalty (negative reward) for possibly running into an edge is more than compensated for by the expected gain for possibly stumbling onto A or B.

The Bellman equation must hold for each state for the value function $v_\pi$ shown in the figure (right). Show numerically that this equation holds for the center state, valued at +0.7, with respect to its four neighboring states, valued at +2.3, +0.4, -0.4, and +0.7. (These numbers are accurate only to one decimal place.)


#### Your answer here here:
...

We need to prove that the **Bellman Equation** holds true for the center square (Value $= 0.7$).

### The Formula
The value of a state is the average of the immediate reward plus the discounted value of the next state.

$$V(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a) [r + \gamma V(s')]$$

### Given Data
* **Current State Value ($V(s)$):** $0.7$
* **Discount factor ($\gamma$):** $0.9$
* **Policy ($\pi$):** Random. The agent picks North, South, East, West with equal probability ($0.25$ each).
* **Immediate Reward ($r$):** $0$ (Because moving between internal white squares gives no reward).
* **Neighbor Values ($V(s')$):**
    * North: $+2.3$
    * South: $+0.4$ (approx)
    * West: $-0.4$
    * East: $+0.7$ (approx)

> **Note:** I am assigning the values from your list ($2.3, 0.4, -0.4, 0.7$) to the directions based on the visual logic of the grid (North is close to A, so it's high; West is close to the edge, so it's low).

### The Calculation
We calculate the expected value by averaging the outcome of the 4 moves.

$$V(center) = 0.25 \times [ (0 + 0.9 \cdot 2.3) + (0 + 0.9 \cdot 0.7) + (0 + 0.9 \cdot 0.4) + (0 + 0.9 \cdot (-0.4)) ]$$

Let's factor out the $0.9$:

$$V(center) = 0.25 \times 0.9 \times [ 2.3 + 0.7 + 0.4 - 0.4 ]$$

Sum the neighbor values:
$$2.3 + 0.7 + 0.4 - 0.4 = 3.0$$

Now complete the multiplication:
$$V(center) = 0.25 \times 0.9 \times 3.0$$
$$V(center) = 0.25 \times 2.7$$
$$V(center) = 0.675$$

### Conclusion
$$0.675 \approx 0.7$$

The calculated value ($0.675$) rounds up to $0.7$, which matches the number in the center of the grid. The Bellman equation holds.